# Lesson 02 - मायक्रोसॉफ्ट एजंट फ्रेमवर्क एक्सप्लोर करणे

**मायक्रोसॉफ्ट एजंट फ्रेमवर्क (MAF)** हा AI एजंट तयार करण्यासाठी एक एकत्रित फ्रेमवर्क आहे. हा चार मुख्य घटकांसह स्वच्छ, संयोज्य आर्किटेक्चर प्रदान करतो:

- **क्लायंट** – AI मॉडेल एंडपॉइंटशी जोडतो आणि संवाद हाताळतो
- **एजंट** – सूचनांसह आणि टूल परिभाषांसह क्लायंटचे व्रॅपिंग करतो
- **टूल्स** – एजंटच्या क्षमता वाढवतात अशा कस्टम फंक्शन्स ज्यांना मॉडेल कॉल करू शकते
- **सेशन** – मल्टी-टर्न संवादांसाठी संभाषणाचा इतिहास ठेवतो

या धड्यात, आपण या संकल्पनांचा वापर करून **प्रवास बुकिंग एजंट** तयार करणार आहोत जो गंतव्यस्थानाची उपलब्धता तपासतो.


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## एजंट फ्रेमवर्क आर्किटेक्चर समजून घेणे

मायक्रोसॉफ्ट एजंट फ्रेमवर्क एक स्तरित आर्किटेक्चर अनुसरतो:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लायंट** – एक `AzureAIProjectAgentProvider` Azure OpenAI डिप्लॉयमेंटशी कनेक्ट करतो. तो प्रमाणीकरण, विनंती स्वरूपन आणि प्रतिसाद पार्सिंग हाताळतो.
2. **एजंट** – क्लायंटमधून `provider.create_agent()` वापरून तयार केला जातो, एजंट मॉडेल प्रवेश, सूचना (सिस्टम प्रॉम्प्ट) आणि साधने एकत्र करतो.
3. **साधने** – `@tool` ने सजवलेल्या Python फंक्शन्स ज्या एजंट क्रिया करण्यासाठी किंवा डेटा प्राप्त करण्यासाठी कॉल करू शकतो.
4. **सेशन** – एक `AgentSession` ऑब्जेक्ट (एजंटद्वारे `agent.create_session()` वापरून तयार केलेले) जे संवाद इतिहास साठवते, जेणेकरून एजंट मागील संदर्भ आठवू शकतो आणि मल्टी-टर्न संवाद सक्षम होतो.

चला प्रत्येक स्तर टप्प्याटप्प्याने तयार करूया.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसह टूल्स जोडणे

टूल्स एजंटला मजकूर तयार करण्यापलीकडे क्रिया करण्याची परवानगी देतात. `@tool` डेकोरेटर एक सामान्य Python फंक्शन एका अशा गोष्टीत रूपांतरित करतो ज्याला एजंट कॉल करू शकतो.

महत्त्वाचे मुद्दे:
- `Annotated[type, "description"]` वापरा जेणेकरून मॉडेलला प्रत्येक पॅरामीटर समजेल.
- डॉकस्ट्रिंग हा टूलचे वर्णन बनतो जे मॉडेल पाहते.
- `approval_mode="never_require"` म्हणजे टूल वापरकर्त्याच्या पुष्टीशिवाय आपोआप चालवले जाते.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## साधने असलेला एजंट तयार करणे

आता आपण क्लायंट, सूचना, आणि साधनांना एकत्र करून एजंट तयार करतो. `सूचना` सिस्टीम प्रॉम्प्ट म्हणून काम करतात — त्या एजंटच्या व्यक्तिमत्व आणि वर्तणुकीची व्याख्या करतात.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रांसह बहु-वळण संवाद

एक `AgentSession` (`agent.create_session()` द्वारे तयार केलेले) संभाषणातील सर्व संदेशांचे ट्रॅक ठेवते. प्रत्येक `agent.run()` कॉलसाठी समान सत्र देऊन, एजंटला पूर्ण संभाषण इतिहासाचा प्रवेश मिळतो आणि तो आधीच्या संदेशांकडे परत पाहू शकतो.

आम्ही `tools=[check_destination_availability]` पास करून एजंटला प्रत्येक वळणात आमचा उपलब्धता तपासणी करणारा कॉल करण्यास परवानगी देतो.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework चे चार स्तंभ एक्सप्लोर केले:

| संकल्पना | तुम्ही काय शिकलात |
|---------|------------------|
| **क्लायंट** | `AzureAIProjectAgentProvider` प्रमाणपत्र-आधारित ऑथने Azure OpenAI शी कनेक्ट करते |
| **एजंट** | `provider.create_agent()` मॉडेल कनेक्शनला सूचनांसह आणि नावासह बांधतो |
| **टूल्स** | `@tool` डेकोरेटर एजंटला कॉल करण्यासाठी Python फंक्शन्स उघडतो |
| **सत्र** | `agent.create_session()` अनेक वळणांमध्ये संभाषणाचा इतिहास ठेवतो |

हे बिल्डिंग ब्लॉक्स एकत्र करून असे एजंट तयार करतात जे नैसर्गिक संभाषणे टिकवू शकतात, बाह्य फंक्शन्स कॉल करू शकतात, आणि संदर्भ जपू शकतात — जे पुढील धडा मध्ये अधिक प्रगत एजंटिक पॅटर्नसाठी पाया तयार करतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेचा प्रयत्न करत असलो तरी, कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा गैरसमज होऊ शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास व्यावसायिक मानव अनुवाद घेणे शिफारसीय आहे. या अनुवादाच्या वापरामुळे होणाऱ्या कोणत्याही गैरसमजुतींसाठी किंवा चुकीच्या अर्थसाधनेसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
